Load and Preprocess

In [4]:
import pandas as pd
import re

df = pd.read_csv("poems-100.csv")

texts = df["text"].dropna().astype(str).tolist()

def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

texts = [clean(t) for t in texts]

Tokenize and build vocabulary

In [5]:
from collections import Counter

tokens = []
for t in texts:
    tokens.extend(t.split())

counter = Counter(tokens)

vocab = sorted(counter.keys())
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

vocab_size = len(vocab)
print("Vocab size:", vocab_size)

Vocab size: 5439


In [6]:
sequence_length = 5

X = []
Y = []

for t in texts:
    words = t.split()
    for i in range(len(words) - sequence_length):
        seq = words[i:i+sequence_length]
        target = words[i+sequence_length]

        X.append([word2idx[w] for w in seq])
        Y.append(word2idx[target])


ONE-HOT ENCODING


In [7]:
import torch
import torch.nn.functional as F

X_tensor = torch.tensor(X)

X_onehot = F.one_hot(X_tensor, num_classes=vocab_size).float()
Y_tensor = torch.tensor(Y)


In [8]:
import torch.nn as nn

class OneHotRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = nn.LSTM(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out


In [18]:
import time

start_time = time.time()
loss_history_onehot = []

onehot_model = OneHotRNN(vocab_size, 128)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(onehot_model.parameters(), lr=0.001)

batch_size = 64

for epoch in range(10):
    total_loss = 0

    for i in range(0, len(X_onehot), batch_size):
        xb = X_onehot[i:i+batch_size]
        yb = Y_tensor[i:i+batch_size]

        pred = onehot_model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (len(X_onehot) / batch_size)
    loss_history_onehot.append(avg_loss)

    print(f"[One-hot] Epoch {epoch+1}, Loss: {avg_loss:.4f}")

end_time = time.time()

onehot_time = end_time - start_time

[One-hot] Epoch 1, Loss: 7.1067
[One-hot] Epoch 2, Loss: 6.5501
[One-hot] Epoch 3, Loss: 6.3727
[One-hot] Epoch 4, Loss: 6.2341
[One-hot] Epoch 5, Loss: 6.1987
[One-hot] Epoch 6, Loss: 6.0001
[One-hot] Epoch 7, Loss: 5.7952
[One-hot] Epoch 8, Loss: 5.6238
[One-hot] Epoch 9, Loss: 5.4168
[One-hot] Epoch 10, Loss: 5.2033


Trainable Word Embeddings

In [10]:
X_idx = torch.tensor(X)
Y_idx = torch.tensor(Y)


In [11]:
class EmbeddingRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out


In [19]:
import time

start_time = time.time()
loss_history_embed = []

embedding_model = EmbeddingRNN(vocab_size, 100, 128)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(embedding_model.parameters(), lr=0.001)

batch_size = 64

for epoch in range(10):
    total_loss = 0

    for i in range(0, len(X_idx), batch_size):
        xb = X_idx[i:i+batch_size]
        yb = Y_idx[i:i+batch_size]

        pred = embedding_model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (len(X_idx) / batch_size)
    loss_history_embed.append(avg_loss)

    print(f"[Embedding] Epoch {epoch+1}, Loss: {avg_loss:.4f}")

end_time = time.time()

embedding_time = end_time - start_time

[Embedding] Epoch 1, Loss: 7.0818
[Embedding] Epoch 2, Loss: 6.3467
[Embedding] Epoch 3, Loss: 5.9623
[Embedding] Epoch 4, Loss: 5.5831
[Embedding] Epoch 5, Loss: 5.2105
[Embedding] Epoch 6, Loss: 4.8600
[Embedding] Epoch 7, Loss: 4.5480
[Embedding] Epoch 8, Loss: 4.2226
[Embedding] Epoch 9, Loss: 3.8109
[Embedding] Epoch 10, Loss: 3.4496


Generate Text

In [13]:
import random

def generate_text(model, start_words, num_words=20, use_embedding=True):
    model.eval()

    words = start_words.copy()

    for _ in range(num_words):
        seq = [word2idx[w] for w in words[-sequence_length:]]

        x = torch.tensor([seq])

        if not use_embedding:
            x = F.one_hot(x, num_classes=vocab_size).float()

        with torch.no_grad():
            out = model(x)
            next_id = torch.argmax(out, dim=1).item()

        words.append(idx2word[next_id])

    return " ".join(words)


In [25]:
print("One-hot model generated text:")
print(
    generate_text(onehot_model,
                  ["the","sun","is","so","bright"],
                  use_embedding=False)
)

One-hot model generated text:
the sun is so bright the heart of the sun of the sky of the sky of the sky of the sky of the sky


In [26]:
print("\nEmbedding model generated text:")
print(
    generate_text(embedding_model,
                  ["the","sun","is","so","bright"],
                  use_embedding=True)
)


Embedding model generated text:
the sun is so bright and i know that the moon of the earth and the sky of the moon emerging hath awakend a heart


In [23]:
print(f"One-hot model training time: {onehot_time:.2f} seconds")
print(f"One-hot model final loss: {loss_history_onehot[-1]:.4f}")

print(f"\nEmbedding model training time: {embedding_time:.2f} seconds")
print(f"Embedding model final loss: {loss_history_embed[-1]:.4f}")

One-hot model training time: 455.35 seconds
One-hot model final loss: 5.2033

Embedding model training time: 54.58 seconds
Embedding model final loss: 3.4496
